In [23]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
df = pd.read_csv(url)
print(df.head())


      lead_source       industry employment_status       location  \
0  organic_search     technology          employed         europe   
1    social_media     technology          employed  south_america   
2        referral         retail          employed         europe   
3        paid_ads  manufacturing           student            NaN   
4        referral  manufacturing           student  north_america   

   annual_income  number_of_courses_viewed  interaction_count  lead_score  \
0        88160.0                         3                  4        0.64   
1        72688.0                         3                  7        0.72   
2        44697.0                         3                  4        0.58   
3            NaN                         3                  6        0.57   
4        24062.0                         4                  5        0.62   

   converted  
0          1  
1          1  
2          1  
3          0  
4          1  


In [14]:
industry_mode = df["industry"].mode()

numerical_features = [
    'interaction_count',
    'lead_score',
    'annual_income',
    'number_of_courses_viewed'
]

correlation_matrix = df[numerical_features].corr()


print(industry_mode)
print(correlation_matrix)

0    technology
Name: industry, dtype: str
                          interaction_count  lead_score  annual_income  \
interaction_count                  1.000000    0.928712       0.106375   
lead_score                         0.928712    1.000000       0.231396   
annual_income                      0.106375    0.231396       1.000000   
number_of_courses_viewed           0.721609    0.769352       0.137542   

                          number_of_courses_viewed  
interaction_count                         0.721609  
lead_score                                0.769352  
annual_income                             0.137542  
number_of_courses_viewed                  1.000000  


In [19]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [20]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

df_train = df_train.drop('converted', axis=1)
df_val = df_val.drop('converted', axis=1)
df_test = df_test.drop('converted', axis=1)

In [21]:
df_train = df_train.fillna('_')
df_val = df_val.fillna('_')
df_test = df_test.fillna('_')

categorical = ['industry','location','lead_source','employment_status']

for col in categorical:
    score = mutual_info_score(df_train[col], y_train)
    print(f"{col} score: {round(score, 2)}")

industry score: 0.0
location score: 0.0
lead_source score: 0.03
employment_status score: 0.02


In [24]:
df_train['annual_income'] = df_train['annual_income'].replace('_', 0).astype(float)
df_val['annual_income'] = df_val['annual_income'].replace('_', 0).astype(float)

In [25]:
categorical = ['industry','location','lead_source','employment_status']
numerical = ['annual_income','number_of_courses_viewed','interaction_count','lead_score']
full_features = categorical + numerical

train_dicts = df_train[full_features].to_dict(orient='records')
val_dicts = df_val[full_features].to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
accuracy = accuracy_score(y_val, y_pred)

print(f"Validation Accuracy: {round(accuracy, 2)}")

Validation Accuracy: 0.65


In [33]:
features_to_check = ['lead_source', 'number_of_courses_viewed', 'interaction_count']

for feature in features_to_check:
    subset_features = [f for f in full_features if f != feature]
    
    train_dicts_sub = df_train[subset_features].to_dict(orient='records')
    val_dicts_sub = df_val[subset_features].to_dict(orient='records')
    
    dv_sub = DictVectorizer(sparse=False)
    X_train_sub = dv_sub.fit_transform(train_dicts_sub)
    X_val_sub = dv_sub.transform(val_dicts_sub)
    
    model_sub = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model_sub.fit(X_train_sub, y_train)
    
    acc_sub = accuracy_score(y_val, model_sub.predict(X_val_sub))
    difference = accuracy - acc_sub
    
    print(f"Excluding '{feature}':")
    print(f"  New Accuracy = {acc_sub}")
    print(f"  Difference   = {difference}\n")

Excluding 'lead_source':
  New Accuracy = 0.642
  Difference   = 0.0030000000000000027

Excluding 'number_of_courses_viewed':
  New Accuracy = 0.643
  Difference   = 0.0020000000000000018

Excluding 'interaction_count':
  New Accuracy = 0.601
  Difference   = 0.04400000000000004



In [34]:
C_values = [0.000001, 0.00001, 0.0001, 0.001]

for c in C_values:
    model_c = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model_c.fit(X_train, y_train)
    
    acc_c = accuracy_score(y_val, model_c.predict(X_val))
    
    print(f"C = {c:<8} -> Validation Accuracy = {round(acc_c, 3)}")

C = 1e-06    -> Validation Accuracy = 0.598
C = 1e-05    -> Validation Accuracy = 0.598
C = 0.0001   -> Validation Accuracy = 0.613
C = 0.001    -> Validation Accuracy = 0.645
